In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
from pathlib import Path
from torch.utils.data import DataLoader

join = os.path.join
import torch

from skimage import io, transform
import torch.nn.functional as F

from pytorch_ood.detector import Mahalanobis
from pytorch_ood.utils import OODMetrics

import monai
from monai.metrics import DiceMetric

from PIL import Image

from peft import LoraConfig, get_peft_model

In [3]:
PROJECT_ROOT=Path("/home/jovyan/thesis_project")
PROJECT_FM_THESIS= PROJECT_ROOT / "FM_thesis"

DATA_ROOT = Path("/home/jovyan/thesis_project/datasets/INbreast/INbreast Release 1.0")
INbreast_masses= DATA_ROOT / "MassCases_cropped"
INbreast_nomasses= DATA_ROOT / "NoMassCases_cropped"

MEDSAM_IMAGE_SIZE=(1024,1024)
VERSAMAMMO_IMAGE_SIZE=(224,224)
MAMMOFM_IMAGE_SIZE=512

MEDSAM_ROOT= PROJECT_FM_THESIS / "MedSAM"
VERSAMAMMO_ROOT = PROJECT_FM_THESIS / "VersaMammo"
VERSAMAMMO_DETECTION_ROOT = VERSAMAMMO_ROOT / "downstream" / "Detection"

WEIGHTS_PATH= PROJECT_ROOT / "models"
MEDSAM_ZEROSHOT = WEIGHTS_PATH / "medsam_vit_b.pth"
MEDSAM_LORA_ZGT= WEIGHTS_PATH / "medsam_LoRA_ZGT_from_gt_masks_fold4.pth"
VERSAMAMMO_ZEROSHOT = WEIGHTS_PATH / "VersaMammo_pretrained" / "VersaMammo (Enb5).pth"

MAMMOFM_ROOT= PROJECT_FM_THESIS / "Mammo-FM"
MAMMOFM_CODEBASE=MAMMOFM_ROOT / "src" / "codebase"
MAMMOFM_CHECKPOINT=PROJECT_ROOT / "models" /"Mammo-FM_BatmanlabTrained_CLIP.tar"

OUTPUT_FEATURES=PROJECT_FM_THESIS / "feature_space_inspection"
OUTPUT_FEATURES.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"


In [4]:
for path in [PROJECT_FM_THESIS, MEDSAM_ROOT, VERSAMAMMO_DETECTION_ROOT, MAMMOFM_CODEBASE]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))



from root_utils.dep_injection_util import (
        BaseDataset,
        NpzLoader,
        PngLoader,
        ConnectedComponentsBBoxFromMask
)

from MedSAM.segment_anything import sam_model_registry
from MedSAM.utils.medSAM_architecture import MedSAM, MedSAMPreprocess
from VersaMammo.models.image_encoder import build_vit_model
from Detectors.retinanet.efficient_net import EfficientNet


xFormers is not available (SwiGLU)
xFormers is not available (Attention)
xFormers is not available (Block)


## MedSAM

In [5]:


sam_model = sam_model_registry["vit_b"](checkpoint=MEDSAM_ZEROSHOT)

model = MedSAM(
    image_encoder=sam_model.image_encoder,
    mask_decoder=sam_model.mask_decoder,
    prompt_encoder=sam_model.prompt_encoder,
).to(DEVICE)


# config = LoraConfig(
#     r=8,
#     lora_alpha=8,
#     target_modules=["qkv"],
#     lora_dropout=0.1,
#     bias="none"
# )
# model = get_peft_model(model, config)


checkpoint = torch.load(MEDSAM_ZEROSHOT, map_location=DEVICE)
model.load_state_dict(checkpoint)

model.eval()

MedSAM(
  (image_encoder): ImageEncoderViT(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): Linear(in_features=3072, out_features=768, bias=True)
          (act): GELU(approximate='none')
        )
      )
    )
    (neck): Sequential(
      (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): LayerNorm2d()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (3): LayerNorm2d()
   

In [10]:
png_loader = PngLoader(dtype=np.uint8)

npz_loader = NpzLoader(dtype=None)

bbox_generator = ConnectedComponentsBBoxFromMask(
    annotation_threshold=0.5,
    allow_empty_mask=True,
)


# id_dataset = BaseDataset(
#     root=DATA_ROOT,
#     format_loader=loader,
#     file_collector=collect_ZGT_files,
#     collector_kwargs={"metadata_df": df_masses},
#     bbox_generator=bbox_generator,
#     transforms=MedSAMPreprocess(MEDSAM_IMAGE_SIZE)
# )





ood_INbreast_MedSAM_dataset = BaseDataset(
    root=INbreast_nomasses,
    format_loader=npz_loader,
    file_collector=None,
    bbox_generator=bbox_generator,
    transforms=MedSAMPreprocess(MEDSAM_IMAGE_SIZE),
    label=-1
)

In [11]:
# Pass ID data through encoder and fit Mahalanobis distance metric
feats_list_id = []
labels_list_id = []


with torch.no_grad():
    for sample in ood_INbreast_MedSAM_dataset:
        imgs = sample["image"].to(DEVICE).unsqueeze(0)   #unsqueeze only if this is not using the batch loader
        y = sample["label"].to(DEVICE)

        z = model.image_encoder(imgs)              # [B, C, H, W] feature maps
        z = z.mean(dim=(2, 3))                     # [B, C] global average pool

        feats_list_id.append(z)
        labels_list_id.append(y)

Z_id = torch.cat(feats_list_id, dim=0)
Y_id = torch.stack(labels_list_id)


In [12]:
Z_id.shape

torch.Size([237, 256])

In [13]:
torch.save(Z_id.detach().cpu(), OUTPUT_FEATURES / "medSAM_zs_INbreast_nomasses")

## VersaMammo encoder features for OOD detection


In [14]:
class VersaMammoEncoderPreprocess:
    """Preprocess images for the VersaMammo ViT-B/14 encoder.

    The encoder is built with img_size=224 and returns a C x 16 x 16 patch feature map.
    We keep this transform image-only because the OOD pipeline only needs encoder features.
    """

    def __init__(self, image_size=(224, 224), mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)):
        self.image_size = image_size
        self.mean = torch.tensor(mean, dtype=torch.float32).view(3, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(3, 1, 1)

    def __call__(self, sample):
        image = sample["image"]
        image = np.asarray(image)

        if image.ndim == 3:
            if image.shape[-1] == 1:
                image = image[..., 0]
            elif image.shape[-1] == 3:
                image = image.mean(axis=-1)
            elif image.shape[0] in (1, 3):
                image = image.mean(axis=0)
            else:
                raise ValueError(f"Unexpected image shape for VersaMammo preprocessing: {image.shape}")

        image = image.astype(np.float32)
        image_min = image.min()
        image_max = image.max()
        image = (image - image_min) / max(image_max - image_min, 1e-8)

        image = torch.from_numpy(image).float().unsqueeze(0).repeat(3, 1, 1)
        image = F.interpolate(
            image.unsqueeze(0),
            size=self.image_size,
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)
        image = (image - self.mean) / self.std

        sample["image"] = image
        return sample


def pool_encoder_features(features):
    if isinstance(features, (tuple, list)):
        features = features[0]
    if features.ndim == 4:
        return features.mean(dim=(2, 3))
    if features.ndim == 3:
        return features.mean(dim=1)
    if features.ndim == 2:
        return features
    return features.flatten(start_dim=1)


def extract_encoder_features(dataset, encoder, device):
    feats_list = []
    labels_list = []

    encoder.eval()
    with torch.no_grad():
        for sample in dataset:
            imgs = sample["image"].to(device).unsqueeze(0)
            labels_list.append(sample["label"].to(device))

            features = encoder(imgs)
            features = pool_encoder_features(features)
            feats_list.append(features)

    return torch.cat(feats_list, dim=0), torch.stack(labels_list)


In [15]:
if not VERSAMAMMO_ZEROSHOT.exists():
    raise FileNotFoundError(
        f"VersaMammo encoder checkpoint not found: {VERSAMAMMO_ZEROSHOT}. "
        "Update VERSAMAMMO_ZEROSHOT to the teacher_checkpoint_*.pth file you want to use."
    )

versamammo_encoder, versamammo_num_features = build_vit_model(
    arch="vit_base",
    pretrained=True,
    pretrained_path=str(VERSAMAMMO_ZEROSHOT),
)
versamammo_encoder = versamammo_encoder.to(DEVICE).eval()
print(f"Loaded VersaMammo ViT encoder with {versamammo_num_features} features.")


Parameter module.image_encoder._conv_stem.weight not found in the model state_dict.
Parameter module.image_encoder._bn0.weight not found in the model state_dict.
Parameter module.image_encoder._bn0.bias not found in the model state_dict.
Parameter module.image_encoder._bn0.running_mean not found in the model state_dict.
Parameter module.image_encoder._bn0.running_var not found in the model state_dict.
Parameter module.image_encoder._bn0.num_batches_tracked not found in the model state_dict.
Parameter module.image_encoder._blocks.0._depthwise_conv.weight not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn1.weight not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn1.bias not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn1.running_mean not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn1.running_var not found in the model state_dict.
Parameter module.image_encoder._blocks.0._bn

In [20]:
png_loader = PngLoader(dtype=np.uint8)

npz_loader = NpzLoader(dtype=None)

bbox_generator = ConnectedComponentsBBoxFromMask(
    annotation_threshold=0.5,
    allow_empty_mask=True,
)


# versamammo_id_dataset = BaseDataset(
#     root=DATA_ROOT,
#     format_loader=loader,
#     file_collector=collect_ZGT_files,
#     collector_kwargs={"metadata_df": df_masses},
#     bbox_generator=bbox_generator,
#     transforms=VersaMammoEncoderPreprocess(VERSAMAMMO_IMAGE_SIZE),
# )



ood_INbreast_VersaMammo_dataset = BaseDataset(
    root=INbreast_masses,
    format_loader=npz_loader,
    file_collector=None,
    bbox_generator=bbox_generator,
    transforms=VersaMammoEncoderPreprocess(VERSAMAMMO_IMAGE_SIZE),
    label=-1,
)



In [21]:
Z_ood_versamammo, _ = extract_encoder_features(
    ood_INbreast_VersaMammo_dataset,
    versamammo_encoder,
    DEVICE,
)

In [22]:
Z_ood_versamammo.shape

torch.Size([108, 768])

In [23]:
torch.save(Z_ood_versamammo.detach().cpu(), OUTPUT_FEATURES / "VersaMammo_INBreast_masses")

## MammoFM encoder features

In [27]:
class MammoFMPreprocess:
    """
    Preprocess image for Mammo-FM / EfficientNet encoder.

    Input sample:
        sample["image"]: np.ndarray or torch.Tensor
            Accepted shapes:
                (H, W)
                (H, W, 3)
                (3, H, W)

    Output:
        sample["image"]: torch.FloatTensor, shape (3, size, size)
    """

    def __init__(
        self,
        size=512,
        mean=0.3089279,
        std=0.25053555408335154,
        eps=1e-8,
    ):
        self.size = int(size)
        self.mean = float(mean)
        self.std = float(std)
        self.eps = float(eps)

    def __call__(self, sample):
        image = sample["image"]

        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image)

        image = image.float()

        # Convert image to CHW format.
        if image.ndim == 2:
            # (H, W) -> (1, H, W)
            image = image.unsqueeze(0)

        elif image.ndim == 3:
            if image.shape[0] in (1, 3):
                # Already (C, H, W)
                pass
            elif image.shape[-1] in (1, 3):
                # (H, W, C) -> (C, H, W)
                image = image.permute(2, 0, 1)
            else:
                raise ValueError(f"Unexpected 3D image shape: {tuple(image.shape)}")

        else:
            raise ValueError(f"Expected 2D or 3D image, got shape: {tuple(image.shape)}")

        # Ensure RGB-like 3-channel input.
        if image.shape[0] == 1:
            image = image.repeat(3, 1, 1)
        elif image.shape[0] != 3:
            raise ValueError(f"Expected 1 or 3 channels, got: {image.shape[0]}")

        # Resize to Mammo-FM detector/encoder input size.
        image = F.interpolate(
            image.unsqueeze(0),
            size=(self.size, self.size),
            mode="bilinear",
            align_corners=False,
        ).squeeze(0)

        # Per-image min-max normalization to [0, 1].
        image = image - image.min()
        image = image / (image.max() + self.eps)

        # Mammo-FM normalization.
        image = (image - self.mean) / self.std

        sample["image"] = image
        sample["resized_size"] = (self.size, self.size)

        return sample

In [25]:
encoder = EfficientNet.from_name(
    "efficientnet-b5",
    override_params={"num_classes": 1},  # avoid relying on classifier head
)

ckpt = torch.load(MAMMOFM_CHECKPOINT, map_location=DEVICE, weights_only=False)["model"]
state_dict = {
    k.replace("image_encoder.", "", 1): v
    for k, v in ckpt.items()
    if k.startswith("image_encoder.")
}

state_dict.pop("_fc.weight", None)
state_dict.pop("_fc.bias", None)

encoder.load_state_dict(state_dict, strict=False)
encoder.source_layer_indexes = [26, 37]  # b5; use [15, 21] for b2
encoder=encoder.to(DEVICE)
encoder.eval()

efficientnet-b5
GlobalParams(batch_norm_momentum=0.99, batch_norm_epsilon=0.001, dropout_rate=0.4, num_classes=1, width_coefficient=1.6, depth_coefficient=2.2, depth_divisor=8, min_depth=None, drop_connect_rate=0.2, image_size=456)


EfficientNet(
  (_conv_stem): Conv2dStaticSamePadding(
    3, 48, kernel_size=(3, 3), stride=(2, 2), bias=False
    (static_padding): ZeroPad2d((0, 1, 0, 1))
  )
  (_bn0): BatchNorm2d(48, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
  (_blocks): ModuleList(
    (0): MBConvBlock(
      (_depthwise_conv): Conv2dStaticSamePadding(
        48, 48, kernel_size=(3, 3), stride=[1, 1], groups=48, bias=False
        (static_padding): ZeroPad2d((1, 1, 1, 1))
      )
      (_bn1): BatchNorm2d(48, eps=0.001, momentum=0.010000000000000009, affine=True, track_running_stats=True)
      (_se_reduce): Conv2dStaticSamePadding(
        48, 12, kernel_size=(1, 1), stride=(1, 1)
        (static_padding): Identity()
      )
      (_se_expand): Conv2dStaticSamePadding(
        12, 48, kernel_size=(1, 1), stride=(1, 1)
        (static_padding): Identity()
      )
      (_project_conv): Conv2dStaticSamePadding(
        48, 24, kernel_size=(1, 1), stride=(1, 1), bias=False
  

In [26]:
png_loader = PngLoader(dtype=np.uint8)

npz_loader = NpzLoader(dtype=None)

bbox_generator = ConnectedComponentsBBoxFromMask(
    annotation_threshold=0.5,
    allow_empty_mask=True,
)




INbreast_masses_dataset = BaseDataset(
    root=INbreast_masses,
    format_loader=npz_loader,
    file_collector=None,
    bbox_generator=bbox_generator,
    transforms=MammoFMPreprocess(MAMMOFM_IMAGE_SIZE),
    label=-1,
)


INbreast_nomasses_dataset = BaseDataset(
    root=INbreast_nomasses,
    format_loader=npz_loader,
    file_collector=None,
    bbox_generator=bbox_generator,
    transforms=MammoFMPreprocess(MAMMOFM_IMAGE_SIZE),
    label=-1,
)



In [28]:
feats_list_id = []
labels_list_id = []


with torch.no_grad():
    for sample in INbreast_masses_dataset:
        imgs = sample["image"].to(DEVICE).unsqueeze(0)
        y = sample["label"].to(DEVICE)

        z, _, _ = encoder.extract_features(imgs)   # [B, C, H, W]

        z = z.mean(dim=(2, 3))                     # [B, C]
        z = z.detach()

        feats_list_id.append(z.cpu())              # keep Mahalanobis fitting on CPU
        labels_list_id.append(y.cpu())

Z = torch.cat(feats_list_id, dim=0)             # [N, C]
Y = torch.stack(labels_list_id).long().view(-1) # [N]

torch.save(Z.detach().cpu(), OUTPUT_FEATURES / "MammoFM_INbreast_allmasses")

In [29]:
feats_list_id = []
labels_list_id = []


with torch.no_grad():
    for sample in INbreast_nomasses_dataset:
        imgs = sample["image"].to(DEVICE).unsqueeze(0)
        y = sample["label"].to(DEVICE)

        z, _, _ = encoder.extract_features(imgs)   # [B, C, H, W]

        z = z.mean(dim=(2, 3))                     # [B, C]
        z = z.detach()

        feats_list_id.append(z.cpu())              # keep Mahalanobis fitting on CPU
        labels_list_id.append(y.cpu())

Z = torch.cat(feats_list_id, dim=0)             # [N, C]
Y = torch.stack(labels_list_id).long().view(-1) # [N]

torch.save(Z.detach().cpu(), OUTPUT_FEATURES / "MammoFM_INbreast_nomasses")

In [17]:
Z.shape

torch.Size([237, 2048])